# Station Stacking v14 - KMIA

Experimental notebook for `KMIA`.

This version keeps the v11 remaining-warmup target and Huber/ridge stack, computes v13 weather aggregates, trains only on a curated v11-plus-weather allowlist, and writes artifacts to `data/calibration/station_stacking_v14`.


In [ ]:
from pathlib import Path
import os
import sys
import warnings

warnings.filterwarnings("ignore", message="IProgress not found.*")
warnings.filterwarnings("ignore", message="Skipping features without any observed values.*")

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "src" / "calibration" / "station_stacking.py").exists():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError("Could not find project root containing src/calibration/station_stacking.py")
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

os.environ["WEATHER_RESEARCH_INCLUDE_DIRECT_NBM"] = "1"

STATION_ID = "KMIA"
PROVIDERS = ("gfs", "hrrr", "nbm")
TREND_COLUMNS = [
    "observed_temp_change_last_1h_f",
    "observed_temp_change_last_3h_f",
    "observed_morning_warmup_rate_f_per_hour",
    "observed_high_so_far_change_since_9am_f",
]
TIMING_MODE = "same_day_11am_live_safe"
FAST_MODE = False
OPTUNA_TRIALS = 30
STACK_OPTUNA_TRIALS = 30
OPTUNA_STARTUP_TRIALS = 15
STACK_OPTUNA_STARTUP_TRIALS = 15
OPTUNA_METRIC = "mae_f"
OPTUNA_VERBOSE = True
MODEL_VERSION = "station_high_regressor_v14_curated_weather_stack"
PROJECT_ROOT


In [ ]:
import numpy as np
import pandas as pd

from src.export_station_stacking_v2_models import export_station_model_weights
from src.calibration.station_stacking import (
    StationStackingConfig,
    V14_DROPPED_FEATURE_COLUMNS,
    V14_FEATURE_COLUMNS,
    YEAR_SPLIT_EXPANDING_FOLDS,
    missing_model_dependencies,
    provider_availability,
    run_station_year_split_experiment,
)


## V14 Contract

`feature_version="v14"` keeps the v11 remaining-warmup, Huber, and ridge-stack lineage, then adds only curated aggregate weather features that pass coverage.


In [ ]:
fold_spec = pd.DataFrame(
    [
        {
            "fold": fold.name,
            "train_start_year": fold.train_start_year,
            "train_end_year": fold.train_end_year,
            "validation_year": fold.validation_year,
        }
        for fold in YEAR_SPLIT_EXPANDING_FOLDS
    ]
)

fold_spec


In [ ]:
V14_FEATURE_COLUMNS, sorted(V14_DROPPED_FEATURE_COLUMNS)


## Data Availability


In [ ]:
availability = provider_availability(
    PROJECT_ROOT,
    timing_mode=TIMING_MODE,
    providers=PROVIDERS,
)

availability.loc[availability["station_id"].eq(STATION_ID)]


## Model Scores


In [ ]:
missing_packages = missing_model_dependencies()
if missing_packages:
    raise ImportError(
        "Missing station-stacking ML packages: "
        + ", ".join(missing_packages)
        + ". Install them with: python -m pip install -r requirements.txt"
    )

config = StationStackingConfig(
    station_id=STATION_ID,
    project_root=PROJECT_ROOT,
    timing_mode=TIMING_MODE,
    providers=PROVIDERS,
    fast_mode=FAST_MODE,
    optuna_trials=OPTUNA_TRIALS,
    stack_optuna_trials=STACK_OPTUNA_TRIALS,
    optuna_startup_trials=OPTUNA_STARTUP_TRIALS,
    stack_optuna_startup_trials=STACK_OPTUNA_STARTUP_TRIALS,
    optuna_metric=OPTUNA_METRIC,
    optuna_verbose=OPTUNA_VERBOSE,
    feature_version="v14",
    target_mode="remaining_warmup",
    base_model_methods=("xgboost", "lightgbm", "catboost"),
    stack_enabled=True,
    hyperparameter_space="wide",
    year_split_folds=YEAR_SPLIT_EXPANDING_FOLDS,
    year_split_test_train_years=(2021, 2025),
    year_split_test_year=2026,
    output_dir=PROJECT_ROOT / "data" / "calibration" / "station_stacking_v14",
)

config.resolved_optuna_storage_path()


In [ ]:
result = run_station_year_split_experiment(config)
result.scoreboard


In [ ]:
exported_weights = export_station_model_weights(
    project_root=PROJECT_ROOT,
    station_id=STATION_ID,
    artifact_dir=config.resolved_output_dir(),
    model_version=MODEL_VERSION,
    timing_mode=config.timing_mode,
    providers=tuple(config.providers),
    feature_version=config.effective_feature_version,
    optuna_metric=config.effective_optuna_metric,
    target_mode=config.effective_target_mode,
    base_model_methods=tuple(config.effective_base_model_methods),
    stack_enabled=config.stack_enabled,
    source_pipeline="notebooks/station_stacking_v14",
)

exported_weights.bundle_path, exported_weights.manifest_path


## Rain-Day V11 vs V14 MAE


In [ ]:
def _rain_day_flags(features: pd.DataFrame) -> pd.DataFrame:
    work = features.copy()
    work["contract_date"] = pd.to_datetime(work["contract_date"]).dt.strftime("%Y-%m-%d")
    rain_signal = pd.Series(False, index=work.index)
    candidate_cols = [
        "observed_is_raining_at_as_of",
        "v4_observed_precip_any",
        "v4_any_forecast_precip",
        "gfs_forecast_has_precip",
        "hrrr_forecast_has_precip",
        "nbm_forecast_has_precip",
    ]
    for column in candidate_cols:
        if column in work:
            rain_signal |= pd.to_numeric(work[column], errors="coerce").fillna(0).gt(0)
    amount_cols = [
        column
        for column in work.columns
        if column.endswith("forecast_precip_total_mm")
        or column.endswith("forecast_precip_max_1h_mm")
        or column in {"observed_precip_recent_at_as_of", "precip_amount"}
    ]
    for column in amount_cols:
        rain_signal |= pd.to_numeric(work[column], errors="coerce").fillna(0).gt(0.01)
    return work.loc[rain_signal, ["contract_date"]].drop_duplicates()


def _prediction_mae_by_method(predictions: pd.DataFrame, version: str, rainy_dates: pd.DataFrame) -> pd.DataFrame:
    if predictions.empty:
        return pd.DataFrame(columns=["version", "method", "rain_day_count", "mae_f", "rmse_f"])
    pred = predictions.copy()
    pred["contract_date"] = pd.to_datetime(pred["contract_date"]).dt.strftime("%Y-%m-%d")
    pred = pred.merge(rainy_dates, on="contract_date", how="inner")
    pred = pred.dropna(subset=["actual_high_f", "predicted_high_f"])
    if pred.empty:
        return pd.DataFrame(columns=["version", "method", "rain_day_count", "mae_f", "rmse_f"])
    pred["error_f"] = pred["actual_high_f"] - pred["predicted_high_f"]
    return (
        pred.groupby("method", dropna=False)
        .agg(
            rain_day_count=("contract_date", "nunique"),
            prediction_count=("contract_date", "size"),
            mae_f=("error_f", lambda s: float(np.mean(np.abs(s)))),
            rmse_f=("error_f", lambda s: float(np.sqrt(np.mean(np.square(s))))),
        )
        .reset_index()
        .assign(version=version)
        [["version", "method", "rain_day_count", "prediction_count", "mae_f", "rmse_f"]]
    )


v11_pred_path = PROJECT_ROOT / "data" / "calibration" / "station_stacking_v11" / f"{STATION_ID}_year_split_test_predictions.csv"
v11_feature_path = PROJECT_ROOT / "data" / "calibration" / "station_stacking_v11" / f"{STATION_ID}_features.csv"
v11_predictions = pd.read_csv(v11_pred_path) if v11_pred_path.exists() else pd.DataFrame()
rain_feature_frame = result.features if not result.features.empty else pd.read_csv(v11_feature_path)
rainy_dates = _rain_day_flags(rain_feature_frame)

rain_mae_comparison = pd.concat(
    [
        _prediction_mae_by_method(v11_predictions, "v11", rainy_dates),
        _prediction_mae_by_method(result.test_predictions, "v14", rainy_dates),
    ],
    ignore_index=True,
)
rain_mae_comparison = rain_mae_comparison.sort_values(["method", "version"]).reset_index(drop=True)
rain_mae_comparison


In [ ]:
rain_mae_wide = rain_mae_comparison.pivot_table(
    index="method",
    columns="version",
    values="mae_f",
    aggfunc="first",
)
if {"v11", "v14"}.issubset(rain_mae_wide.columns):
    rain_mae_wide["v14_minus_v11_mae_f"] = rain_mae_wide["v14"] - rain_mae_wide["v11"]
rain_mae_wide.sort_values("v14_minus_v11_mae_f" if "v14_minus_v11_mae_f" in rain_mae_wide else rain_mae_wide.columns[0])


## V14 Feature Coverage


In [ ]:
v14_feature_coverage = (
    result.features[V14_FEATURE_COLUMNS]
    .notna()
    .mean()
    .mul(100)
    .sort_values(ascending=False)
    .rename("coverage_pct")
    .reset_index()
    .rename(columns={"index": "feature"})
)

v14_feature_coverage


In [ ]:
result.feature_columns.loc[result.feature_columns["feature"].isin(V14_FEATURE_COLUMNS)]


## Dropped Feature Check


In [ ]:
dropped_present = result.feature_columns.loc[
    result.feature_columns["feature"].isin(V14_DROPPED_FEATURE_COLUMNS)
]

dropped_present


## Morning Trend Coverage


In [ ]:
trend_coverage = (
    result.features[TREND_COLUMNS]
    .notna()
    .mean()
    .mul(100)
    .sort_values(ascending=False)
    .rename("coverage_pct")
    .reset_index()
    .rename(columns={"index": "feature"})
)

trend_coverage


## Rounded Within 1F Accuracy


In [ ]:
preds = pd.concat(
    [
        result.validation_predictions.assign(period="validation_2024_2025"),
        result.test_predictions.assign(period="oof_2026"),
    ],
    ignore_index=True,
)

predicted_high = pd.to_numeric(preds["predicted_high_f"], errors="coerce")
preds["predicted_high_rounded_f"] = np.floor(predicted_high + 0.5)
preds["within_1f_after_round"] = (
    pd.to_numeric(preds["actual_high_f"], errors="coerce") - preds["predicted_high_rounded_f"]
).abs().le(1)

within_1f_accuracy_by_period = (
    preds
    .dropna(subset=["actual_high_f", "predicted_high_rounded_f"])
    .groupby(["period", "method"], as_index=False)
    .agg(
        count=("within_1f_after_round", "size"),
        within_1f_count=("within_1f_after_round", "sum"),
        within_1f_accuracy_pct=("within_1f_after_round", lambda x: x.mean() * 100),
    )
    .sort_values(["period", "within_1f_accuracy_pct"], ascending=[True, False])
)

within_1f_accuracy_by_period


## Version Comparison


In [ ]:
comparison_frames = []
for version, folder in [
    ("v1", PROJECT_ROOT / "data" / "calibration" / "station_stacking"),
    ("v2", PROJECT_ROOT / "data" / "calibration" / "station_stacking_v2"),
    ("v3", PROJECT_ROOT / "data" / "calibration" / "station_stacking_v3"),
    ("v4", PROJECT_ROOT / "data" / "calibration" / "station_stacking_v4"),
    ("v5", PROJECT_ROOT / "data" / "calibration" / "station_stacking_v5"),
    ("v6", PROJECT_ROOT / "data" / "calibration" / "station_stacking_v6"),
    ("v7", PROJECT_ROOT / "data" / "calibration" / "station_stacking_v7"),
    ("v8", PROJECT_ROOT / "data" / "calibration" / "station_stacking_v8"),
    ("v9", PROJECT_ROOT / "data" / "calibration" / "station_stacking_v9"),
    ("v10", PROJECT_ROOT / "data" / "calibration" / "station_stacking_v10"),
    ("v11", PROJECT_ROOT / "data" / "calibration" / "station_stacking_v11"),
    ("v13", PROJECT_ROOT / "data" / "calibration" / "station_stacking_v13"),
    ("v14", PROJECT_ROOT / "data" / "calibration" / "station_stacking_v14"),
]:
    path = folder / f"{STATION_ID}_year_split_scoreboard.csv"
    if path.exists():
        frame = pd.read_csv(path)
        frame["version"] = version
        comparison_frames.append(frame)

version_comparison = pd.concat(comparison_frames, ignore_index=True) if comparison_frames else pd.DataFrame()
if not version_comparison.empty:
    version_comparison = version_comparison.sort_values(["period", "mae_f", "version", "method"]).reset_index(drop=True)
version_comparison


## 2026 OOF Weather Brackets


In [ ]:
result.bracket_metrics
